# Part 2: Warranty Satisfaction Analysis
## Aspect-Based Sentiment Analysis for Product Warranties

**Goal:** Find reviews mentioning warranty/guarantee and calculate satisfaction scores

**Creative Approaches Used:**
1. 🔥 **Word Embeddings** - Using pre-trained GloVe to find semantically similar words
2. 🔥 **Automated Keyword Expansion** - Finding misspellings and synonyms automatically
3. 🔥 **Product-Level Aggregation** - Computing warranty satisfaction per product
4. 🔥 **Comparative Analysis** - Comparing warranty ratings vs overall ratings

---

## 1️⃣ Setup & Load Data

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
# Load preprocessed data
print("Loading preprocessed data...")

train_df = pd.read_csv("clean_dataset/train_clean.csv")
val_df = pd.read_csv("clean_dataset/val_clean.csv")

print(f"✓ Train: {train_df.shape}")
print(f"✓ Val: {val_df.shape}")

# Combine for complete analysis
full_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"✓ Combined: {full_df.shape}")

# Preview
full_df.head()



Loading preprocessed data...
✓ Train: (747564, 13)
✓ Val: (82921, 13)
✓ Combined: (830485, 13)


,overall,text_clean,verified,vote_count,asin,review_len_chars,review_len_words,summary_len,asin_freq,style_freq,style_encoded,review_year,review_month
0,2,cannot learn. i have an older urc wr7 remote a...,0,0.0,0511189877,0.051164,0.100134,-1.162506,-1.250161,1.042888,0,2016,11
1,5,zero programming needed! miracle!?. first time...,1,0.0,0511189877,-0.269520,-0.183425,-0.695187,-1.250161,1.042888,0,2016,6
2,4,work good and program easy.. got them and only...,1,0.0,0511189877,-0.977336,-0.923828,-0.461527,-1.250161,1.042888,0,2016,3
3,5,same a twc remote. i got tired of the remote b...,1,0.0,0511189877,-0.749101,-0.624516,-0.695187,-1.250161,1.042888,0,2016,1
4,5,good quality cord. after purchasing cheap cord...,1,0.0,0594459451,-0.855996,-0.829308,-0.928847,-1.250161,1.042888,0,2016,10


---

## 2️⃣ Load Word Embeddings (CREATIVE APPROACH #1)

### 🔥 Why This is Creative:
Instead of just searching for exact words like "warranty" and "guarantee", we use **word embeddings** to find semantically similar words!

**Word embeddings** are vector representations of words where similar words have similar vectors.

**Example:**
- `warranty` → finds: guaranty, warrantee, coverage, protection
- `guarantee` → finds: assurance, warranty, promise

This catches:
- ✅ Synonyms (coverage, protection)
- ✅ Misspellings (warrantee, garantee)
- ✅ Related terms (refund, replacement)

In [4]:
# Load pre-trained word embeddings
import gensim.downloader as api

print("Loading GloVe embeddings (this takes ~1 minute first time)...")
print("GloVe = Global Vectors for Word Representation")
print("These are pre-trained on billions of words from Wikipedia!\n")

try:
    # Load 100-dimensional GloVe embeddings
    word_vectors = api.load("glove-wiki-gigaword-100")
    print("✓ Successfully loaded GloVe embeddings")
    print(f"  Vocabulary size: {len(word_vectors):,} words")
    has_embeddings = True
except Exception as e:
    print(f"⚠️ Could not load embeddings: {e}")
    print("Will use rule-based approach instead")
    has_embeddings = False

Loading GloVe embeddings (this takes ~1 minute first time)...
GloVe = Global Vectors for Word Representation
These are pre-trained on billions of words from Wikipedia!

✓ Successfully loaded GloVe embeddings
  Vocabulary size: 400,000 words


---

## 3️⃣ Find Similar Words Using Embeddings (CREATIVE APPROACH #2)

### 🔥 Why This is Creative:
We **automatically expand** our keyword list using semantic similarity!

**Traditional Approach:**
```python
keywords = ['warranty', 'guarantee']  # Only 2 words!
```

**Our Creative Approach:**
```python
# Start with 2 base words
# → Find 20 similar words for each
# → Add common misspellings
# → Result: 50+ keywords automatically!
```

In [5]:
# Base keywords
base_keywords = ['warranty', 'guarantee']

# Start with base keywords
warranty_keywords = set(base_keywords)

print("🔍 Finding similar words using word embeddings...\n")

🔍 Finding similar words using word embeddings...



In [6]:
# Find similar words using embeddings
if has_embeddings:
    for keyword in base_keywords:
        try:
            # Find 20 most similar words
            # Returns: [(word, similarity_score), ...]
            similar_words = word_vectors.most_similar(keyword, topn=20)
            
            print(f"Similar words to '{keyword}':")
            print(f"{'Word':<20}{'Similarity Score':<20}")
            print("-" * 40)
            
            for word, score in similar_words[:10]:
                print(f"{word:<20}{score:<20.3f}")
                
                # Add words with similarity > 0.55
                # (0.55 is our threshold - you can adjust this!)
                if score > 0.55:
                    warranty_keywords.add(word.lower())
            
            print()
            
        except KeyError:
            print(f"'{keyword}' not found in vocabulary")
else:
    print("Skipping embedding-based similarity (not available)")

Similar words to 'warranty':
Word                Similarity Score    
----------------------------------------
warranties          0.795               
renewals            0.548               
lease               0.529               
powertrain          0.512               
end-user            0.490               
exclusivity         0.490               
redundancy          0.486               
expiration          0.481               
annuity             0.480               
termination         0.479               

Similar words to 'guarantee':
Word                Similarity Score    
----------------------------------------
guarantees          0.893               
guaranteed          0.865               
ensure              0.792               
guaranteeing        0.754               
assure              0.740               
necessary           0.722               
ensuring            0.716               
insure              0.711               
secure              0.706             

In [7]:
# Add common misspellings and variations (rule-based)
# These are words we know people misspell!

common_variations = [
    # Spelling mistakes
    'warrenty', 'warrantee', 'warenty', 'warrany', 'warrnty',
    'garantee', 'guarentee', 'guarante', 'gaurantee', 'garentee',
    
    # Related terms
    'warrantied', 'warranties', 'guaranteed', 'guarantees',
    'coverage', 'covered', 'protection', 'replace', 'replacement',
    'refund', 'return', 'defect', 'defective', 'broken',
    
    # Warranty-related phrases
    'warranty period', 'warranty claim', 'warranty service',
    'manufacturer warranty', 'extended warranty', 'limited warranty'
]

warranty_keywords.update(common_variations)

print(f"\n✓ Total warranty-related keywords: {len(warranty_keywords)}")
print(f"\nFirst 30 keywords:")
print(sorted(list(warranty_keywords))[:30])


✓ Total warranty-related keywords: 40

First 30 keywords:
['adequate', 'assure', 'broken', 'coverage', 'covered', 'defect', 'defective', 'ensure', 'ensuring', 'extended warranty', 'garantee', 'garentee', 'gaurantee', 'guarante', 'guarantee', 'guaranteed', 'guaranteeing', 'guarantees', 'guarentee', 'insure', 'limited warranty', 'manufacturer warranty', 'necessary', 'protection', 'refund', 'replace', 'replacement', 'return', 'secure', 'warenty']


---

## 4️⃣ Detect Warranty Mentions in Reviews

Now we search for ANY of our keywords in each review!

In [8]:
# Detect warranty mentions
print("Scanning reviews for warranty mentions...\n")

# Create empty list to store results
mentions_warranty = []

# Check each review
for text in full_df['text_clean']:
    # Convert to string and lowercase
    if pd.isna(text) or not isinstance(text, str):
        mentions_warranty.append(False)
        continue
    
    text_lower = text.lower()
    
    # Check if ANY keyword appears in text
    found = False
    for keyword in warranty_keywords:
        if keyword in text_lower:
            found = True
            break
    
    mentions_warranty.append(found)

# Add to dataframe
full_df['mentions_warranty'] = mentions_warranty

# Get warranty reviews
warranty_reviews = full_df[full_df['mentions_warranty']].copy()

print(f"✓ Found {len(warranty_reviews):,} reviews mentioning warranty")
print(f"  That's {len(warranty_reviews)/len(full_df)*100:.2f}% of all reviews!\n")

# Distribution by rating
print("Distribution by rating:")
warranty_dist = warranty_reviews['overall'].value_counts().sort_index()
print(f"{'Rating':<10}{'Count':<10}{'Percentage':<15}")
print("-" * 35)
for rating, count in warranty_dist.items():
    pct = count / len(warranty_reviews) * 100
    print(f"{rating}★{'':<8}{count:<10}{pct:<14.1f}%")

Scanning reviews for warranty mentions...

✓ Found 200,504 reviews mentioning warranty
  That's 24.14% of all reviews!

Distribution by rating:
Rating    Count     Percentage     
-----------------------------------
1★        35367     17.6          %
2★        20346     10.1          %
3★        22154     11.0          %
4★        34503     17.2          %
5★        88134     44.0          %


---

## 5️⃣ Load Product Mapping & Add ASIN

In [9]:
# Try to load product information
try:
    product_df = pd.read_csv("dataset/title_brand.csv")
    print(f"✓ Loaded product mapping: {len(product_df)} products")
    has_product_info = True
except FileNotFoundError:
    print("⚠️ Product mapping not found")
    has_product_info = False

✓ Loaded product mapping: 786445 products


In [10]:
# Load original data to get ASIN (product IDs)
print("Loading original data to get product IDs...")
original_train = pd.read_csv("dataset/train_data.csv")

# Add ASIN to our dataframe
full_df['asin'] = original_train['asin'][:len(full_df)]

print(f"✓ Added ASIN for {full_df['asin'].notna().sum()} reviews")

Loading original data to get product IDs...
✓ Added ASIN for 830485 reviews


---

## 6️⃣ Calculate Warranty Satisfaction by Product (CREATIVE APPROACH #3)

### 🔥 Why This is Creative:
Instead of just showing individual reviews, we **aggregate by product** to get product-level insights!

**What we calculate for each product:**
1. Average warranty rating
2. Number of warranty reviews
3. Standard deviation (consistency)
4. Total stars

This tells us:
- ✅ Which products have warranty problems
- ✅ Which products have excellent warranties
- ✅ Which products have consistent warranty experiences

In [11]:
# Filter to warranty reviews with ASIN
warranty_with_asin = full_df[full_df['mentions_warranty'] & full_df['asin'].notna()].copy()

print(f"Reviews with warranty mentions and product ID: {len(warranty_with_asin)}")

Reviews with warranty mentions and product ID: 200504


In [12]:
# Group by product and calculate statistics
print("\nCalculating warranty satisfaction by product...\n")

warranty_by_product = warranty_with_asin.groupby('asin').agg({
    'overall': ['mean', 'std', 'count', 'sum']  # Multiple aggregations!
}).round(2)

# Rename columns for clarity
warranty_by_product.columns = ['avg_warranty_rating', 'std_warranty_rating', 
                                'warranty_review_count', 'total_stars']

# Filter: only products with at least 3 warranty reviews
# (Need enough reviews to be reliable!)
min_reviews = 3
warranty_by_product = warranty_by_product[
    warranty_by_product['warranty_review_count'] >= min_reviews
]

# Sort by average rating (highest first)
warranty_by_product = warranty_by_product.sort_values('avg_warranty_rating', ascending=False)

print(f"✓ Found {len(warranty_by_product)} products with ≥{min_reviews} warranty reviews")
print(f"\nStatistics:")
print(warranty_by_product.describe().round(2))


Calculating warranty satisfaction by product...

✓ Found 17026 products with ≥3 warranty reviews

Statistics:
       avg_warranty_rating  std_warranty_rating  warranty_review_count  \
count             17026.00             17026.00               17026.00   
mean                  3.60                 1.37                   9.19   
std                   0.79                 0.56                  13.82   
min                   1.00                 0.00                   3.00   
25%                   3.12                 1.03                   3.00   
50%                   3.67                 1.50                   5.00   
75%                   4.17                 1.73                   9.00   
max                   5.00                 2.31                 461.00   

       total_stars  
count     17026.00  
mean         33.08  
std          49.95  
min           3.00  
25%          12.00  
50%          18.00  
75%          32.00  
max        1603.00  


In [13]:
# Add product information if available
if has_product_info:
    # Merge with product info
    warranty_by_product = warranty_by_product.merge(
        product_df[['asin', 'title', 'brand']], 
        left_index=True, 
        right_on='asin', 
        how='left'
    )
    print("✓ Added product titles and brands")
else:
    print("⚠️ Product info not available - will show ASIN codes only")

✓ Added product titles and brands


---

## 7️⃣ Top & Bottom Products by Warranty Satisfaction

In [14]:
# Top 10 products
print("="*90)
print("TOP 10 PRODUCTS BY WARRANTY SATISFACTION")
print("="*90)

top_10 = warranty_by_product.head(10)

if has_product_info:
    print(f"\n{'Rank':<6}{'Brand':<20}{'Product':<35}{'Avg★':<8}{'Count':<8}{'ASIN':<15}")
    print("-" * 92)
    
    for idx, row in enumerate(top_10.itertuples(), 1):
        brand = str(row.brand if hasattr(row, 'brand') else 'Unknown')[:18]
        title = str(row.title if hasattr(row, 'title') else 'Unknown')[:33]
        avg_rating = row.avg_warranty_rating
        count = int(row.warranty_review_count)
        asin = row.asin if hasattr(row, 'asin') else row.Index
        
        print(f"{idx:<6}{brand:<20}{title:<35}{avg_rating:<8.2f}{count:<8}{asin:<15}")
else:
    print(f"\n{'Rank':<6}{'ASIN':<15}{'Avg Rating':<12}{'Review Count':<15}{'Std Dev':<10}")
    print("-" * 58)
    
    for idx, (asin, row) in enumerate(top_10.iterrows(), 1):
        print(f"{idx:<6}{asin:<15}{row['avg_warranty_rating']:<12.2f}"
              f"{int(row['warranty_review_count']):<15}{row['std_warranty_rating']:<10.2f}")

TOP 10 PRODUCTS BY WARRANTY SATISFACTION

Rank  Brand               Product                            Avg★    Count   ASIN           
--------------------------------------------------------------------------------------------
1     rooCASE             rooCASE Samsung Galaxy Tab S 10.5  5.00    4       B00L2N5LO4     
2     Aluratek            Aluratek AIRMM03F Wi-Fi Internet   5.00    4       B016GNX30S     
3     UGREEN              UGREEN Ethernet Cable Cat7 Cat 6   5.00    4       B00QV1F0TS     
4     IdeaNext            Ideanext Bullet Camera, 1080P WiF  5.00    3       B01ANMNTOS     
5     MobilePal           MobilePal ProofFit 1.7-Inch Heads  5.00    3       B016HKUSEO     
6     Onlier               Onlier Compatible Charger and He  5.00    3       B001ODDBSU     
7     Adesso              Adesso iMouse E10 - Vertical Ergo  5.00    3       B00GN0WQBW     
8     Acer                Acer XB240H Abpr 24-Inch Full HD   5.00    3       B00QS0AK6U     
9     Coolerguys           C

In [15]:
# Bottom 10 products
print("\n" + "="*90)
print("BOTTOM 10 PRODUCTS BY WARRANTY SATISFACTION")
print("="*90)

bottom_10 = warranty_by_product.tail(10)

if has_product_info:
    print(f"\n{'Rank':<6}{'Brand':<20}{'Product':<35}{'Avg★':<8}{'Count':<8}{'ASIN':<15}")
    print("-" * 92)
    
    for idx, row in enumerate(bottom_10.itertuples(), 1):
        brand = str(row.brand if hasattr(row, 'brand') else 'Unknown')[:18]
        title = str(row.title if hasattr(row, 'title') else 'Unknown')[:33]
        avg_rating = row.avg_warranty_rating
        count = int(row.warranty_review_count)
        asin = row.asin if hasattr(row, 'asin') else row.Index
        
        rank = len(warranty_by_product) - len(warranty_by_product) + idx
        print(f"{rank:<6}{brand:<20}{title:<35}{avg_rating:<8.2f}{count:<8}{asin:<15}")
else:
    print(f"\n{'Rank':<6}{'ASIN':<15}{'Avg Rating':<12}{'Review Count':<15}{'Std Dev':<10}")
    print("-" * 58)
    
    for idx, (asin, row) in enumerate(bottom_10.iterrows(), 1):
        print(f"{idx:<6}{asin:<15}{row['avg_warranty_rating']:<12.2f}"
              f"{int(row['warranty_review_count']):<15}{row['std_warranty_rating']:<10.2f}")


BOTTOM 10 PRODUCTS BY WARRANTY SATISFACTION

Rank  Brand               Product                            Avg★    Count   ASIN           
--------------------------------------------------------------------------------------------
1     HXYPPY              2 in 1 Lighting Adapter to Charge  1.00    3       B011RJOBEI     
2     Toshiba             Toshiba Satellite Fusion 15 L55W-  1.00    3       B01B3JA1XC     
3     BestElec            Micro USB Retractable Cable, Best  1.00    3       B014QT4B6Y     
4     JBtek               JBtek All Black Sleeved PWM Fan S  1.00    3       B00OZ10FI2     
5     DIMPLES EXCEL       Dimples Excel 2 in 1 Precision Di  1.00    3       B01B7X4HR0     
6     ALLie               ALLie Home 360 Degree Camera 24/7  1.00    3       B01B82OCP2     
7     binj                binj XS Tablet for Android         1.00    3       B01BCTWQRC     
8     Dell                Dell 452-BBTR Latitude E7250 E745  1.00    3       B01BJ2ILIA     
9     Tenfly            

---

## 8️⃣ Visualizations

In [16]:
# Create comprehensive visualization
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Color scheme
colors = ['#d62728', '#ff7f0e', '#ffbb78', '#98df8a', '#2ca02c']

print("Generating visualizations...")

Generating visualizations...


<Figure size 2000x1200 with 0 Axes>

In [17]:

# 1. Distribution of warranty ratings
ax1 = fig.add_subplot(gs[0, :2])

# Use matplotlib's hist instead of pandas hist
ax1.hist(warranty_by_product['avg_warranty_rating'], 
         bins=30, color='purple', alpha=0.7, edgecolor='black')

# Add mean and median lines
mean_val = warranty_by_product['avg_warranty_rating'].mean()
median_val = warranty_by_product['avg_warranty_rating'].median()

ax1.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
ax1.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')

ax1.set_xlabel('Average Warranty Rating', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Products', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Warranty Satisfaction by Product', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

print("✓ Chart 1 created")

✓ Chart 1 created


In [18]:
# 2. Warranty mentions by rating
ax2 = fig.add_subplot(gs[0, 2])

warranty_dist.plot(kind='bar', color=colors, alpha=0.8, edgecolor='black', ax=ax2)
ax2.set_xlabel('Rating', fontsize=11, fontweight='bold')
ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
ax2.set_title('Warranty Mentions\nby Rating', fontsize=12, fontweight='bold')
ax2.set_xticklabels([f'{i}★' for i in warranty_dist.index], rotation=0)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(warranty_dist.values):
    ax2.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

print("✓ Chart 2 created")

✓ Chart 2 created


In [19]:
# 3. Scatter: Review count vs Average rating
ax3 = fig.add_subplot(gs[1, 0])

scatter = ax3.scatter(
    warranty_by_product['warranty_review_count'],
    warranty_by_product['avg_warranty_rating'],
    alpha=0.6, s=50, 
    c=warranty_by_product['avg_warranty_rating'],
    cmap='RdYlGn', vmin=1, vmax=5, 
    edgecolors='black', linewidth=0.5
)

ax3.set_xlabel('Number of Warranty Reviews', fontsize=11, fontweight='bold')
ax3.set_ylabel('Average Rating', fontsize=11, fontweight='bold')
ax3.set_title('Review Count vs Rating', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax3, label='Avg Rating')

print("✓ Chart 3 created")

✓ Chart 3 created


<Figure size 640x480 with 0 Axes>

In [20]:
# 4. Box plot of warranty ratings by star rating
ax4 = fig.add_subplot(gs[1, 1])

# Prepare data for box plot
warranty_reviews_grouped = [
    warranty_reviews[warranty_reviews['overall'] == i]['overall'].values 
    for i in range(1, 6)
]

bp = ax4.boxplot(warranty_reviews_grouped, labels=['1★', '2★', '3★', '4★', '5★'],
                patch_artist=True, showmeans=True)

# Color boxes
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax4.set_xlabel('Rating', fontsize=11, fontweight='bold')
ax4.set_ylabel('Distribution', fontsize=11, fontweight='bold')
ax4.set_title('Warranty Review Rating Distribution', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

print("✓ Chart 4 created")

✓ Chart 4 created


In [21]:
# 5. Top 10 products horizontal bar
ax5 = fig.add_subplot(gs[1, 2])

top_10_sorted = top_10.sort_values('avg_warranty_rating', ascending=True)
y_pos = np.arange(len(top_10_sorted))

if has_product_info:
    labels = [str(row.brand if hasattr(row, 'brand') else 'Unknown')[:12] 
             for row in top_10_sorted.itertuples()]
else:
    labels = [f"Product {i+1}" for i in range(len(top_10_sorted))]

bars = ax5.barh(
    y_pos, 
    top_10_sorted['avg_warranty_rating'].values, 
    color=[plt.cm.RdYlGn(v/5) for v in top_10_sorted['avg_warranty_rating'].values],
    alpha=0.8, 
    edgecolor='black'
)

ax5.set_yticks(y_pos)
ax5.set_yticklabels(labels, fontsize=8)
ax5.set_xlabel('Avg Warranty Rating', fontsize=11, fontweight='bold')
ax5.set_title('Top 10 Products', fontsize=12, fontweight='bold')
ax5.set_xlim([0, 5.5])
ax5.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_10_sorted['avg_warranty_rating'].values):
    ax5.text(v, i, f' {v:.2f}', va='center', fontsize=8)

print("✓ Chart 5 created")

✓ Chart 5 created


### 🔥 CREATIVE APPROACH #4: Overall vs Warranty Rating Comparison

This is a **unique insight** - comparing warranty-specific ratings to overall product ratings!

**Why This Matters:**
- If warranty rating ≈ overall rating → Warranty quality matches product quality
- If warranty rating < overall rating → Warranty is a weak point
- If warranty rating > overall rating → Warranty is a strength!

This helps identify:
- Products where warranty is dragging down satisfaction
- Products where good warranty boosts satisfaction

In [22]:
# # 6. Comparison: Overall vs Warranty ratings (CREATIVE!)
# ax6 = fig.add_subplot(gs[2, :])

# # Calculate overall product ratings (all reviews)
# all_product_ratings = full_df.groupby('asin')['overall'].mean()

# # Create comparison dataframe
# comparison_df = pd.DataFrame({
#     'warranty_rating': warranty_by_product['avg_warranty_rating'],
#     'overall_rating': all_product_ratings
# }).dropna()



# # Scatter plot
# ax6.scatter(comparison_df['overall_rating'], comparison_df['warranty_rating'],
#            alpha=0.5, s=50, edgecolors='black', linewidth=0.5)

# # Add diagonal line (perfect match)
# ax6.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect Match (y=x)')

# ax6.set_xlabel('Overall Product Rating', fontsize=12, fontweight='bold')
# ax6.set_ylabel('Warranty-Specific Rating', fontsize=12, fontweight='bold')
# ax6.set_title('Overall Rating vs Warranty Rating Comparison', fontsize=14, fontweight='bold')
# ax6.legend()
# ax6.grid(alpha=0.3)
# ax6.set_xlim([1, 5])
# ax6.set_ylim([1, 5])

# # Calculate and show correlation
# corr = comparison_df['overall_rating'].corr(comparison_df['warranty_rating'])
# ax6.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
#         transform=ax6.transAxes, fontsize=11, fontweight='bold',
#         verticalalignment='top',
#         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# print("✓ Chart 6 created (CREATIVE INSIGHT!)")
# print(f"  Correlation between overall and warranty ratings: {corr:.3f}")

In [23]:
# 6. Comparison: Overall vs Warranty ratings (CREATIVE!)
ax6 = fig.add_subplot(gs[2, :])

# Calculate overall product ratings (all reviews)
all_product_ratings = full_df.groupby('asin')['overall'].mean()

# Align the indexes - only keep products that exist in both datasets
common_products = warranty_by_product.index.intersection(all_product_ratings.index)

warranty_ratings_aligned = warranty_by_product.loc[common_products, 'avg_warranty_rating']
overall_ratings_aligned = all_product_ratings.loc[common_products]

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'warranty_rating': warranty_ratings_aligned,
    'overall_rating': overall_ratings_aligned
}).dropna()

# Rest of your code remains the same...
ax6.scatter(comparison_df['overall_rating'], comparison_df['warranty_rating'],
           alpha=0.5, s=50, edgecolors='black', linewidth=0.5)

# Add diagonal line (perfect match)
ax6.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect Match (y=x)')

ax6.set_xlabel('Overall Product Rating', fontsize=12, fontweight='bold')
ax6.set_ylabel('Warranty-Specific Rating', fontsize=12, fontweight='bold')
ax6.set_title('Overall Rating vs Warranty Rating Comparison', fontsize=14, fontweight='bold')
ax6.legend()
ax6.grid(alpha=0.3)
ax6.set_xlim([1, 5])
ax6.set_ylim([1, 5])

# Calculate and show correlation
corr = comparison_df['overall_rating'].corr(comparison_df['warranty_rating'])
ax6.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
        transform=ax6.transAxes, fontsize=11, fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

print("✓ Chart 6 created (CREATIVE INSIGHT!)")
print(f"  Correlation between overall and warranty ratings: {corr:.3f}")
print(f"  Number of products in comparison: {len(comparison_df)}")

✓ Chart 6 created (CREATIVE INSIGHT!)
  Correlation between overall and warranty ratings: nan
  Number of products in comparison: 0


In [24]:
# Show all plots
plt.suptitle('Warranty Satisfaction Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.show()

print("\n✓ All visualizations created!")

<Figure size 640x480 with 0 Axes>


✓ All visualizations created!


---

## 9️⃣ Keyword Frequency Analysis

In [25]:
# Extract all words from warranty reviews
print("Analyzing which warranty keywords appear most frequently...\n")

all_warranty_words = []

for text in warranty_reviews['text_clean'].dropna():
    words = text.lower().split()
    all_warranty_words.extend(words)

# Count word frequencies
word_freq = Counter(all_warranty_words)

# Find which of our warranty keywords actually appear
warranty_keyword_freq = {}
for kw in warranty_keywords:
    count = word_freq.get(kw, 0)
    if count > 0:
        warranty_keyword_freq[kw] = count

# Sort by frequency
warranty_keyword_freq = dict(
    sorted(warranty_keyword_freq.items(), key=lambda x: x[1], reverse=True)
)

print("Top 20 Warranty Keywords Found:")
print(f"{'Keyword':<25}{'Frequency':<12}{'% of Warranty Reviews':<25}")
print("-" * 62)

for kw, freq in list(warranty_keyword_freq.items())[:20]:
    pct = (freq / len(warranty_reviews)) * 100
    print(f"{kw:<25}{freq:<12}{pct:<25.2f}%")

Analyzing which warranty keywords appear most frequently...

Top 20 Warranty Keywords Found:
Keyword                  Frequency   % of Warranty Reviews    
--------------------------------------------------------------
replacement              36881       18.39                    %
return                   31338       15.63                    %
replace                  31315       15.62                    %
warranty                 18276       9.12                     %
protection               17095       8.53                     %
secure                   11946       5.96                     %
broken                   9693        4.83                     %
defective                9574        4.77                     %
necessary                6107        3.05                     %
covered                  5238        2.61                     %
refund                   5052        2.52                     %
adequate                 4956        2.47                     %
coverage     

---

## 🔟 Sample Reviews

In [26]:
# Show positive warranty reviews
print("="*70)
print("SAMPLE REVIEWS MENTIONING WARRANTY")
print("="*70)

print("\n🟢 POSITIVE (5 stars) Warranty Reviews:")
print("-" * 70)

positive_warranty = warranty_reviews[warranty_reviews['overall'] == 5]
if len(positive_warranty) > 0:
    samples = positive_warranty.sample(min(3, len(positive_warranty)))
    
    for idx, row in samples.iterrows():
        print(f"\nRating: {int(row['overall'])}★")
        print(f"Text: {row['text_clean'][:200]}...")
        print("-" * 70)

SAMPLE REVIEWS MENTIONING WARRANTY

🟢 POSITIVE (5 stars) Warranty Reviews:
----------------------------------------------------------------------

Rating: 5★
Text: insurance peace of mind for four years.. this is a guarantee system for my new answering machine telephone system. i hope never to need it but having it is cheaper than replacing the system if it fail...
----------------------------------------------------------------------

Rating: 5★
Text: wireless network is now in beast mode. talking about taking your wireless network into beast mode. it wa fast and easy to setup and i replaced an older linksys ea6900. i love the auto frequency select...
----------------------------------------------------------------------

Rating: 5★
Text: great addition to the apple pencil.. i ordered this cable mostly because i wanted the ability to charge my pencil in my ipad at the same time. most of the time i charge my pencil in the lightning port...
----------------------------------------------

In [27]:
# Show negative warranty reviews
print("\n🔴 NEGATIVE (1-2 stars) Warranty Reviews:")
print("-" * 70)

negative_warranty = warranty_reviews[warranty_reviews['overall'] <= 2]
if len(negative_warranty) > 0:
    samples = negative_warranty.sample(min(3, len(negative_warranty)))
    
    for idx, row in samples.iterrows():
        print(f"\nRating: {int(row['overall'])}★")
        print(f"Text: {row['text_clean'][:200]}...")
        print("-" * 70)


🔴 NEGATIVE (1-2 stars) Warranty Reviews:
----------------------------------------------------------------------

Rating: 2★
Text: buggy with just middling sound. this unit replaced the original bose solo that i had for years. the sound over the original is an improvement but nothing earth shattering. the main issue i have had is...
----------------------------------------------------------------------

Rating: 2★
Text: and the worst part is i have to pay for the shipping. i must have returned it since i do not own a 3tb unit and if i remembered it correctly the one i receive wa malfunctioning and the worst part is i...
----------------------------------------------------------------------

Rating: 2★
Text: not sturdy. bought this because another model i purchased i wa unable to access home button while in holder. this one had erratic side closure and my phone popped out when my bike fell and i had to re...
----------------------------------------------------------------------


---

## 1️⃣1️⃣ Save Results

In [30]:
# Save warranty satisfaction by product
warranty_by_product.to_csv("warranty_satisfaction_by_product.csv")
print("✓ Saved: warranty_satisfaction_by_product.csv")

# Save warranty reviews
warranty_reviews[['text_clean', 'overall']].to_csv("warranty_reviews.csv", index=False)
print("✓ Saved: warranty_reviews.csv")

# Save keyword list
with open("warranty_keywords.txt", "w") as f:
    f.write("Warranty-related keywords used:\n\n")
    f.write("\n".join(sorted(warranty_keywords)))
print("✓ Saved: warranty_keywords.txt")

✓ Saved: warranty_satisfaction_by_product.csv
✓ Saved: warranty_reviews.csv
✓ Saved: warranty_keywords.txt


---

## 1️⃣2️⃣ Final Summary & Insights

In [31]:
# Calculate final statistics
print("="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n📊 Overall Statistics:")
print(f"  Total reviews analyzed: {len(full_df):,}")
print(f"  Reviews mentioning warranty: {len(warranty_reviews):,} ({len(warranty_reviews)/len(full_df)*100:.2f}%)")
print(f"  Unique products with warranty mentions: {warranty_reviews['asin'].nunique():,}")
print(f"  Products with ≥{min_reviews} warranty reviews: {len(warranty_by_product):,}")

print(f"\n📈 Warranty Satisfaction Metrics:")
print(f"  Average warranty rating: {warranty_by_product['avg_warranty_rating'].mean():.2f}★")
print(f"  Median warranty rating: {warranty_by_product['avg_warranty_rating'].median():.2f}★")
print(f"  Best warranty rating: {warranty_by_product['avg_warranty_rating'].max():.2f}★")
print(f"  Worst warranty rating: {warranty_by_product['avg_warranty_rating'].min():.2f}★")
print(f"  Standard deviation: {warranty_by_product['avg_warranty_rating'].std():.2f}")

# Calculate quality tiers
good_warranty = sum(warranty_by_product['avg_warranty_rating'] >= 4.0)
poor_warranty = sum(warranty_by_product['avg_warranty_rating'] < 3.0)

print(f"\n💡 Key Insights:")
print(f"  • {good_warranty} products ({good_warranty/len(warranty_by_product)*100:.1f}%) have good warranty satisfaction (≥4.0★)")
print(f"  • {poor_warranty} products ({poor_warranty/len(warranty_by_product)*100:.1f}%) have poor warranty satisfaction (<3.0★)")
print(f"  • Average {warranty_by_product['warranty_review_count'].mean():.1f} warranty reviews per product")
print(f"  • Correlation between overall and warranty ratings: {corr:.3f}")

# Interpret correlation
if corr > 0.7:
    print(f"    → Strong positive correlation: Good warranty ➜ Good overall rating")
elif corr > 0.4:
    print(f"    → Moderate correlation: Warranty affects overall satisfaction")
else:
    print(f"    → Weak correlation: Warranty may not be main factor")

print("\n" + "="*70)
print("PART 2 COMPLETED ✓")
print("="*70)

FINAL SUMMARY

📊 Overall Statistics:
  Total reviews analyzed: 830,485
  Reviews mentioning warranty: 200,504 (24.14%)
  Unique products with warranty mentions: 52,446
  Products with ≥3 warranty reviews: 17,347

📈 Warranty Satisfaction Metrics:
  Average warranty rating: 3.60★
  Median warranty rating: 3.67★
  Best warranty rating: 5.00★
  Worst warranty rating: 1.00★
  Standard deviation: 0.79

💡 Key Insights:
  • 6221 products (35.9%) have good warranty satisfaction (≥4.0★)
  • 3134 products (18.1%) have poor warranty satisfaction (<3.0★)
  • Average 9.2 warranty reviews per product
  • Correlation between overall and warranty ratings: nan
    → Weak correlation: Warranty may not be main factor

PART 2 COMPLETED ✓
